In [1]:
# Add higher directory to python modules path

import sys

sys.path.append("..")

In [2]:
import os
import glob

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from modules.hmm import get_hits, get_seqs

In [3]:
import seaborn as sns
sns.set(font_scale=0.25)

In [4]:
PROJECT_ID = "closing_circuit"

DATA_DIR = "../data/runs/closing-circuit/"
RUN_DIR = os.path.join(
    DATA_DIR,
    "2025-05-16"
)
# TREE_FILE = "aquificota_curated.fasta.treefile"

E_VALUE_THR = 1e-20
ABSENCE_THR = 0.85
CORE_THR    = 0.90

CLUSTERMAP_METHOD = "ward"
CLUSTERMAP_METRIC = "hamming"

SIGNIFICANCE_THR = 0.01

CYCLE_COLOR_MAPPING = {
    "arsenic": "#e8c5aaff",
    "carbon": "#75a56bff",
    "methane": "#e8dfe2ff",
    "nitrogen": "#386b9aff",
    "oxygen": "#b9a3cbff",
    "photosynthesis": "#7fd68bff",
    "sulfur": "#edd970ff",
    "hydrogen": "#8EC3E6",
    "iron": "#861f2bff",
    "selenium": "#a78873ff",
    "transporters": "#F4A261"
}

RANDOM_SEED = 666

## Sites

In [5]:
selected_datasets = [
    "PRJNA288027",
    "PRJNA700657",
    "PRJNA627556",
    "PRJNA362739"
]

In [6]:
sites_df = metadata_df = pd.read_excel(
    os.path.join(
        DATA_DIR,
        "subsurface-metadata.xlsx"
    ),
    sheet_name="sites",
    engine="openpyxl"
)

# There may be multiple datasets per sit
sites_df["dataset"] = sites_df["dataset"].str.split(",")
sites_df = sites_df.explode("dataset")

sites_df = sites_df[sites_df["dataset"].isin(selected_datasets)]

sites_df["site_borehole"] = \
    sites_df["site"] + " - BH-" + sites_df["borehole"].astype(str)

sites_df[["site", "borehole", "country", "dataset"]].sample(10)

,site,borehole,country,dataset
69,Surficial,3026,Canada,PRJNA700657
67,Paskapoo,986,Canada,PRJNA700657
55,Sibbald Valley,123,Canada,PRJNA700657
63,Calgary Valley,760,Canada,PRJNA700657
53,Irvine Valley,114,Canada,PRJNA700657
66,Paskapoo,983,Canada,PRJNA700657
38,Rifle Integrated Field Research site,FP-101,USA,PRJNA288027
3,Äspö HRL,KA2198A,Sweden,PRJNA627556
59,Paskapoo_Horseshoe Canyon,144,Canada,PRJNA700657
37,Rifle Integrated Field Research site,CD-01,USA,PRJNA288027


In [7]:
fig = px.scatter_map(
    sites_df,
    lat=sites_df["latitude"],
    lon=sites_df["longitude"],
    text="site_borehole",
    hover_name="site_borehole",
    zoom=1.5,
    width=900,
    height=750
)

fig.update_traces(
    marker={
        "color": "red",
        "size": 9
    },
    textposition="bottom center",
    textfont_size=11
)

fig.update_layout(
    showlegend=False,
    hovermode="closest",
    map=dict(
        center=go.layout.map.Center(
            lat=52.135573094010754,
            lon=-112.412129117182
        ),
        # bearing=50,
        # pitch=50,
        zoom=6.05
    )
)

fig.show()

In [8]:
fig = px.scatter_map(
    sites_df,
    lat=sites_df["latitude"],
    lon=sites_df["longitude"],
    # color="dataset",
    text="site",
    hover_name="site",
    zoom=1.5,
    width=900,
    height=750
)

fig.update_traces(
    marker={
        "color": "red",
        "size": 9
    },
    textposition="bottom center",
    textfont_size=11
)

fig.update_layout(
    showlegend=False,
    hovermode="closest",
    map=dict(
        center=go.layout.map.Center(
            lat=39.60113454313982,
            lon=-109.16124131102413
        ),
        # bearing=50,
        # pitch=50,
        zoom=7
    )
)

fig.show()

In [9]:
fig = px.scatter_map(
    sites_df,
    lat=sites_df["latitude"],
    lon=sites_df["longitude"],
    text="site",
    hover_name="site",
    zoom=1.5,
    width=900,
    height=750
)

fig.update_traces(
    marker={
        "color": "red",
        "size": 9
    },
    textposition="bottom center",
    textfont_size=11
)

fig.update_layout(
    showlegend=False,
    hovermode="closest",
    map=dict(
        center=go.layout.map.Center(
            lat=60.15224455800149,
            lon=19.907514425326845
        ),
        # bearing=50,
        # pitch=50,
        zoom=5.05
    )
)

fig.show()

## Metadata

In [10]:
# Get correct HMM model name mapping
hmm_mapping = {}

for hmm_path in glob.glob("../data/profiles/metabolic/*.hmm"):
    hmm_filename = os.path.basename(hmm_path)

    with open(hmm_path, mode="r") as handle:
        hmm_name = handle.readlines()[1]\
            .replace("NAME  ", "")\
            .replace("\n", "")

        hmm_mapping[hmm_filename] = hmm_name

In [11]:
metadata_df = pd.read_excel(
    os.path.join(
        DATA_DIR,
        "gene-table-oxidoreductases.xlsx"
    ),
    sheet_name="genes",
    engine="openpyxl"
)

metadata_df = metadata_df\
    .dropna(how="all", axis=0)\
    .dropna(how="all", axis=1)

# Fix HMM names
metadata_df["hmm_model"] = metadata_df["hmm_file"]\
    .map(hmm_mapping)

# Some enzymes may take or yield multiple donors and acceptors simultaneously
redox_cols = [
    "donor_reduced",
    "donor_oxidized",
    "acceptor_oxidized",
    "acceptor_reduced"
]
for col in redox_cols:
    metadata_df[col] = metadata_df[col].fillna("").str.split(",")
    metadata_df = metadata_df.explode(col)
    metadata_df[col] = metadata_df[col].apply(lambda row: "".join(row))
    metadata_df.loc[metadata_df[col] == ""] = np.nan

# Add missing HMM model names
metadata_df["hmm_model"] = metadata_df["hmm_model"]\
    .fillna(metadata_df["hmm_file"])\
    .apply(lambda row: f"{str(row).replace('.hmm', '')}")

metadata_df

,enzyme_name,gene,gene_operon,gene_complex,gene_desc,gene_function,class,cycle,pathway,KO,...,acceptor_oxidized,acceptor_reduced,redox_potential,reversible,hmm_file,hmm_thr,hmm_type,hmm_source,references,hmm_model
0,arsenate reductase (glutaredoxin),arsC,arsRDABC,NaN,NaN,catalytic,terminal,arsenic,Arsenate reduction for detoxification,K00537,...,AsO4,AsO3,NaN,NaN,arsC_glut.hmm,80.0,full,METABOLIC,"METABOLIC,4,11,12,13",arsC_glut
1,arsenate reductase (thioredoxin),arsC,arsRDABC,NaN,NaN,catalytic,terminal,arsenic,Arsenate reduction for detoxification,K03741,...,AsO4,AsO3,NaN,NaN,arsC_thio.hmm,172.0,full,METABOLIC,"METABOLIC,4,11,12,13",arsC_thio
2,arsenate reductase (cytochrome c),aoxA,NaN,NaN,small subunit,NaN,NaN,arsenic,NaN,K08355,...,cytochrome (ox),cytochrome (red),NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
3,arsenate reductase (cytochrome c),aoxB,NaN,NaN,large subunit,NaN,NaN,arsenic,NaN,K08356,...,cytochrome (ox),cytochrome (red),NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
4,arsenite oxidase,aioA,NaN,NaN,NaN,NaN,NaN,arsenic,Arsenite oxidation for detoxification and ener...,K08356,...,cytochrome (ox),cytochrome (red),NaN,NaN,aioA.hmm,800.0,full,METABOLIC,METABOLIC,aioA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
614,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan
616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan


In [12]:
kofam_df = pd.read_table("../data/profiles/kofam/ko_list.tsv")

kofam_df = kofam_df.rename(columns={
    "knum": "KO",
    "threshold": "hmm_thr_kofam",
    "definition": "enzyme_name",
    "score_type": "hmm_type"
})
kofam_df["hmm_model"] = kofam_df["KO"].copy()
kofam_df["hmm_file"] = kofam_df["KO"] + ".hmm"
kofam_df["hmm_thr_kofam"] = kofam_df["hmm_thr_kofam"]\
    .replace("-", np.nan)\
    .astype(float)

metadata_df["hmm_thr"] = metadata_df["hmm_thr"].astype(float)

metadata_df = pd.merge(
    left=metadata_df,
    right=kofam_df[["hmm_file", "hmm_thr_kofam"]],
    on=["hmm_file"],
    how="left"
)

# Update HMM thresholds for KOfam profiles
metadata_df.loc[
    metadata_df["hmm_file"].isin(kofam_df["hmm_file"].unique()),
    "hmm_thr"
] = None
metadata_df["hmm_thr"] = metadata_df["hmm_thr"]\
    .fillna(metadata_df["hmm_thr_kofam"])

# Add missing HMM models
missing_kos = kofam_df[~kofam_df["KO"].isin(metadata_df["hmm_model"])]
metadata_df = pd.concat([
    metadata_df,
    missing_kos[[
        "enzyme_name",
        "hmm_model",
        "hmm_file",
        "hmm_type",
        "hmm_thr_kofam"
    ]]
])

metadata_df

,enzyme_name,gene,gene_operon,gene_complex,gene_desc,gene_function,class,cycle,pathway,KO,...,acceptor_reduced,redox_potential,reversible,hmm_file,hmm_thr,hmm_type,hmm_source,references,hmm_model,hmm_thr_kofam
0,arsenate reductase (glutaredoxin),arsC,arsRDABC,NaN,NaN,catalytic,terminal,arsenic,Arsenate reduction for detoxification,K00537,...,AsO3,NaN,NaN,arsC_glut.hmm,80.0,full,METABOLIC,"METABOLIC,4,11,12,13",arsC_glut,NaN
1,arsenate reductase (thioredoxin),arsC,arsRDABC,NaN,NaN,catalytic,terminal,arsenic,Arsenate reduction for detoxification,K03741,...,AsO3,NaN,NaN,arsC_thio.hmm,172.0,full,METABOLIC,"METABOLIC,4,11,12,13",arsC_thio,NaN
2,arsenate reductase (cytochrome c),aoxA,NaN,NaN,small subunit,NaN,NaN,arsenic,NaN,K08355,...,cytochrome (red),NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN
3,arsenate reductase (cytochrome c),aoxB,NaN,NaN,large subunit,NaN,NaN,arsenic,NaN,K08356,...,cytochrome (red),NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,NaN
4,arsenite oxidase,aioA,NaN,NaN,NaN,NaN,NaN,arsenic,Arsenite oxidation for detoxification and ener...,K08356,...,cytochrome (red),NaN,NaN,aioA.hmm,800.0,full,METABOLIC,METABOLIC,aioA,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27318,glycosylated lysosomal membrane protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,K28024.hmm,NaN,full,NaN,NaN,K28024,152.20
27319,all-trans retinoic acid-induced differentiatio...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,K28025.hmm,NaN,domain,NaN,NaN,K28025,113.50
27320,transmembrane protein 87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,K28026.hmm,NaN,full,NaN,NaN,K28026,339.87
27321,"MFS transporter, DHA1 family, spermine transpo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,K28027.hmm,NaN,full,NaN,NaN,K28027,664.97


## Raw data

In [13]:
hits_path = os.path.join(
    RUN_DIR,
    f"{PROJECT_ID}_hmmer.txt"
)
seqs_path = os.path.join(
    RUN_DIR,
    f"{PROJECT_ID}.faa"
)

hits_df = get_hits(hits_path)
# seqs_df = get_seqs(seqs_path)

# Create MAG and gene caller ID columns
hits_df["mag"] = hits_df["target_name"]\
    .str.split(".").str[0]
hits_df["gene_caller_id"] = hits_df["target_name"]\
    .str.split("_").str[-1]

# Add metadata information
hits_df = pd.merge(
    left=hits_df,
    right=metadata_df,
    left_on="query_name",
    right_on="hmm_model",
    how="left"
)

hits_df

,target_name,target_accession,query_name,query_accession,e_value_full_seq,score_full_seq,bias_full_seq,e_value_best_dom,score_best_dom,bias_best_dom,...,acceptor_reduced,redox_potential,reversible,hmm_file,hmm_thr,hmm_type,hmm_source,references,hmm_model,hmm_thr_kofam
0,GCA_001789195.1_OGJ92339.1,-,K19268,-,1.500000e-185,626.5,0.0,1.800000e-185,626.2,0.0,...,NaN,NaN,NaN,K19268.hmm,NaN,full,NaN,NaN,K19268,250.9
1,GCA_001789205.1_OGJ94580.1,-,K19268,-,1.500000e-185,626.5,0.0,1.800000e-185,626.2,0.0,...,NaN,NaN,NaN,K19268.hmm,NaN,full,NaN,NaN,K19268,250.9
2,GCA_001789275.1_OGK07055.1,-,K19268,-,1.500000e-185,626.5,0.0,1.800000e-185,626.2,0.0,...,NaN,NaN,NaN,K19268.hmm,NaN,full,NaN,NaN,K19268,250.9
3,GCA_030694975.1_MDP3304524.1,-,K19268,-,6.100000e-185,624.5,0.0,7.200000e-185,624.2,0.0,...,NaN,NaN,NaN,K19268.hmm,NaN,full,NaN,NaN,K19268,250.9
4,GCA_018830725.1_MBU2597727.1,-,K19268,-,1.800000e-183,619.6,0.0,2.000000e-183,619.5,0.0,...,NaN,NaN,NaN,K19268.hmm,NaN,full,NaN,NaN,K19268,250.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3722004,GCA_030696345.1_MDP3540285.1,-,K02437,-,8.600000e-06,34.8,5.2,4.800000e-01,19.3,1.1,...,NaN,NaN,NaN,K02437.hmm,NaN,domain,NaN,NaN,K02437,35.7
3722005,GCA_030650685.1_MDO8754807.1,-,K02437,-,9.000000e-06,34.7,0.0,9.800000e-06,34.6,0.0,...,NaN,NaN,NaN,K02437.hmm,NaN,domain,NaN,NaN,K02437,35.7
3722006,GCA_001828795.1_OHC30568.1,-,K02437,-,9.100000e-06,34.7,4.2,9.400000e-02,21.6,0.4,...,NaN,NaN,NaN,K02437.hmm,NaN,domain,NaN,NaN,K02437,35.7
3722007,GCA_001828815.1_OHC39683.1,-,K02437,-,9.100000e-06,34.7,4.2,9.400000e-02,21.6,0.4,...,NaN,NaN,NaN,K02437.hmm,NaN,domain,NaN,NaN,K02437,35.7


In [14]:
set_res = set(hits_df["query_name"].unique()).difference(
    set(metadata_df["hmm_model"].unique())
)

print(f"[!] Number of HMM models not present in metadata: {len(set_res)}")

[!] Number of HMM models not present in metadata: 46


## Preprocessing

In [15]:
# Filter by E-value threshold
hits_df = hits_df[
    hits_df["e_value_full_seq"] <= E_VALUE_THR
]

# Keep a copy without filtering for further plots
hits_df_nobitscore = hits_df.copy()

# Filter by bitscore threshold
hits_df = hits_df[
    hits_df["score_full_seq"] >= hits_df["hmm_thr"]
]

# Get only those hits with the lowest E-value
hits_df = hits_df.loc[
    hits_df.groupby("target_name")["e_value_full_seq"].idxmin()
].reset_index(drop=True)

hits_df

,target_name,target_accession,query_name,query_accession,e_value_full_seq,score_full_seq,bias_full_seq,e_value_best_dom,score_best_dom,bias_best_dom,...,acceptor_reduced,redox_potential,reversible,hmm_file,hmm_thr,hmm_type,hmm_source,references,hmm_model,hmm_thr_kofam
0,GCA_001595385.3_OIJ73184.1,-,TIGR00339,TIGR00339,2.600000e-130,444.0,0.0,3.000000e-130,443.8,0.0,...,Adenyl-SO4,NaN,NaN,TIGR00339.hmm,181.80,full,"METABOLIC,TIGRFAM","METABOLIC,KEGG,TIGRFAM,1,31",TIGR00339,NaN
1,GCA_001595385.3_OIJ73392.1,-,nirS_alignment,-,1.600000e-242,815.2,3.4,1.800000e-242,815.0,3.4,...,NO,0.35,NaN,nitrite_reductase_nirS.hmm,200.00,full,METABOLIC,"METABOLIC,1,39",nirS_alignment,NaN
2,GCA_001595385.3_OIJ73510.1,-,DFE_0463,-,7.200000e-47,169.1,12.5,6.800000e-24,93.6,4.1,...,Fe2+,NaN,NaN,DFE_0463.hmm,100.00,full,METABOLIC,"METABOLIC,52",DFE_0463,NaN
3,GCA_001595385.3_OIJ73639.1,-,TIGR02866,TIGR02866,7.700000e-43,155.6,0.3,9.400000e-43,155.3,0.3,...,H2O,NaN,NaN,TIGR02866.hmm,144.30,full,"METABOLIC,TIGRFAM","METABOLIC,TIGRFAM,31",TIGR02866,NaN
4,GCA_001595385.3_OIJ73671.1,-,TIGR02378,TIGR02378,3.200000e-24,94.2,0.0,3.500000e-24,94.0,0.0,...,NH3,0.34,NaN,TIGR02378.hmm,69.85,full,"METABOLIC,TIGRFAM","METABOLIC,TIGRFAM,39",TIGR02378,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68514,GCF_034523315.1_WP_305250023.1,-,TIGR02374,TIGR02374,1.500000e-253,853.2,0.0,1.700000e-253,853.0,0.0,...,NH3,0.34,NaN,TIGR02374.hmm,442.75,full,"METABOLIC,TIGRFAM","METABOLIC,TIGRFAM,39",TIGR02374,NaN
68515,GCF_034523315.1_WP_305250142.1,-,sulfur_dioxygenase_sdo_alignment,-,8.600000e-87,299.8,0.0,9.800000e-87,299.6,0.0,...,H2O,NaN,NaN,sulfur_dioxygenase_sdo.hmm,120.00,full,METABOLIC,METABOLIC,sulfur_dioxygenase_sdo_alignment,NaN
68516,GCF_034523315.1_WP_305250564.1,-,DmkB,-,1.300000e-42,155.5,0.0,1.600000e-42,155.1,0.0,...,Fe2+,NaN,NaN,DmkB.hmm,25.00,full,METABOLIC,"METABOLIC,51",DmkB,NaN
68517,GCF_034523315.1_WP_305250613.1,-,TIGR00782,TIGR00782,3.300000e-113,387.0,0.5,3.700000e-113,386.8,0.5,...,H2O,NaN,NaN,TIGR00782.hmm,54.05,full,"METABOLIC,TIGRFAM","METABOLIC,TIGRFAM",TIGR00782,NaN


In [ ]:
# hits_df.to_csv(
#     os.path.join(
#         RUN_DIR,
#         "hits-processed.csv"
#     ),
#     index=False
# )

hits_df = pd.read_csv(
    os.path.join(
        RUN_DIR,
        "hits-processed.csv"
    )
)

In [17]:
from pandas.api.types import CategoricalDtype


group_cols = [
    "mag",
    "donor_reduced",
    "donor_oxidized",
    "acceptor_oxidized",
    "acceptor_reduced",
    "cycle"
]

# Merge ETC elements
network_df = hits_df[group_cols]\
    .apply(lambda row: row.str.replace(" (red)", ""))\
    .apply(lambda row: row.str.replace(" (ox)", ""))

network_df = network_df\
    .groupby(group_cols, as_index=False)\
    .size()

# ---------------------------------------------------------------------------- #
# ---------------------------------------------------------------------------- #
# ---------------------------------------------------------------------------- #

network_df["donor"] = network_df["donor_reduced"].copy()
network_df["acceptor"] = network_df["acceptor_oxidized"].copy()

# Rename elements of the ETC and donors and terminal acceptors
node_mapping = {
    "FAD": "ETC",
    "FADH2": "ETC",
    "NAD+": "ETC",
    "NADH": "ETC",
    "ferredoxin": "ETC",
    "cytochrome": "ETC",
    "rusticyanin": "ETC",
    "quinol": "ETC",
    "quinone": "ETC",
    "ubiquinol": "ETC",
    "ubiquinone": "ETC",
    "menaquinol": "ETC",
    "menaquinone": "ETC",
    "thioredoxin": "ETC",
    "thioredoxin disulfide": "ETC",
    "glutaredoxin": "ETC",
    "glutaredoxin disulfide": "ETC",
    "glutathione": "ETC"
}
network_df["donor"] = network_df["donor"].replace(node_mapping)
network_df["acceptor"] = network_df["acceptor"].replace(node_mapping)

# Remove specific nodes
removed_donors = [
    "ATP",
    "H2O"
]

network_df = network_df[~network_df["donor_reduced"].isin(removed_donors)]

# ---------------------------------------------------------------------------- #
# ---------------------------------------------------------------------------- #
# ---------------------------------------------------------------------------- #

all_pairs = network_df["donor"].tolist() + network_df["acceptor"].tolist()
all_pairs = set(all_pairs)

categories = CategoricalDtype(
    categories=all_pairs,
    ordered=True
)
network_df["donor"] = network_df["donor"].astype(categories)
network_df["acceptor"] = network_df["acceptor"].astype(categories)

# ---------------------------------------------------------------------------- #

# Add sample information
site_metadata_df = pd.read_excel(
    os.path.join(
        DATA_DIR,
        "subsurface-metadata.xlsx"
    ),
    sheet_name="mags"
)
site_metadata_df["site_id"] = \
    site_metadata_df["dataset"] + "-" + \
    site_metadata_df["borehole"].astype(str)

site_metadata_df["mag"] = site_metadata_df["assembly"]\
    .str.split(".").str[0]

network_df = pd.merge(
    left=network_df,
    right=site_metadata_df,
    on="mag",
    how="left"
)

# ---------------------------------------------------------------------------- #

network_df

,mag,donor_reduced,donor_oxidized,acceptor_oxidized,acceptor_reduced,cycle,size,donor,acceptor,dataset,site,borehole,assembly,level,wgs,biosample,isolate,taxonomy,sample_id,site_id
0,GCA_001595385,ETC,ETC,Fe3+,Fe2+,iron,1,ETC,Fe3+,PRJNA288027,NaN,NaN,GCA_001595385.3,Contig,LVEI00000000,SAMN04316215,NaN,Deltaproteobacteria bacterium GWC2_55_46,GWC2_55_46,PRJNA288027-nan
1,GCA_001595385,Fe2+,Fe3+,cytochrome,cytochrome,iron,1,Fe2+,ETC,PRJNA288027,NaN,NaN,GCA_001595385.3,Contig,LVEI00000000,SAMN04316215,NaN,Deltaproteobacteria bacterium GWC2_55_46,GWC2_55_46,PRJNA288027-nan
2,GCA_001595385,Fe2+,Fe3+,rusticyanin,rusticyanin,iron,1,Fe2+,ETC,PRJNA288027,NaN,NaN,GCA_001595385.3,Contig,LVEI00000000,SAMN04316215,NaN,Deltaproteobacteria bacterium GWC2_55_46,GWC2_55_46,PRJNA288027-nan
3,GCA_001595385,H2S,polysulfide,quinone,quinol,sulfur,1,H2S,ETC,PRJNA288027,NaN,NaN,GCA_001595385.3,Contig,LVEI00000000,SAMN04316215,NaN,Deltaproteobacteria bacterium GWC2_55_46,GWC2_55_46,PRJNA288027-nan
4,GCA_001595385,NADH,NAD+,Fe3+,Fe2+,iron,2,ETC,Fe3+,PRJNA288027,NaN,NaN,GCA_001595385.3,Contig,LVEI00000000,SAMN04316215,NaN,Deltaproteobacteria bacterium GWC2_55_46,GWC2_55_46,PRJNA288027-nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29736,GCF_034523315,quinol,quinone,NO3,NO2,nitrogen,2,ETC,NO3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29737,GCF_034523315,rusticyanin,rusticyanin,O2,H2O,iron,1,ETC,O2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29738,GCF_034523315,succinate,fumarate,quinone,quinol,"ETC, carbon",2,succinate,ETC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29739,GCF_034523315,thiosulfate,SO4,cytochrome,cytochrome,sulfur,7,thiosulfate,ETC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
network_df[["mag", "dataset"]].drop_duplicates().value_counts("dataset")

dataset
PRJNA288027    1561
PRJNA700657    1289
PRJNA627556     947
PRJNA362739     496
Name: count, dtype: int64

In [19]:
n_mags_real = network_df[["mag", "dataset"]]\
    .drop_duplicates()\
    .value_counts("dataset")\
    .sum()

n_mags_real = int(n_mags_real)

n_mags_hyp = 2661 + 983 + 1278 + 1567

n_mags_ratio = 100 * n_mags_real / n_mags_hyp
n_mags_ratio = round(n_mags_ratio, 2)

n_mags_real, n_mags_hyp, n_mags_ratio

(4293, 6489, 66.16)

In [20]:
network_df[["mag", "level"]].drop_duplicates().value_counts("level")

level
Contig             2291
Scaffold           2001
Complete genome       1
Name: count, dtype: int64

In [21]:
# Color mappings

# ---------------------------------------------------------------------------- #
# Node colors

cycle_mapping = {
    "fermentation": "#545454",
    "ETC, carbon": "#545454",
    "carbon fixation": "green",
    "sulfur": "#fcd400",
    "iron": "#c85739",
    "nitrogen": "#5271ff",
    "ETC, oxygen, iron": "#fa3232",
    "ETC, oxygen, sulfur": "#fa3232",
    "ETC, oxygen": "#fa3232",
    "sulfur, oxygen": "#fa3232",
    "hydrogen": "lightblue",
    "selenium": "plum",
    "arsenic": "darkorange",
    "arsenic, DMSO, manganese, nitrogen, carbon, iron": "violet"
}

cycle_mapping_legend = {
    "Arsenic": "darkorange",
    "Carbon": "#545454",
    "Carbon fixation": "green",
    "ETC": "silver",
    "Hydrogen": "lightblue",
    "Iron": "#c85739",
    "Nitrogen": "#5271ff",
    "Oxygen": "#fa3232",
    "Selenium": "plum",
    "Sulfur": "#fcd400"
}

# ---------------------------------------------------------------------------- #
# Link colors

dataset_mapping = {
    "PRJNA288027": "#0f52ba",
    "PRJNA362739": "#ffa648",
    "PRJNA627556": "#008080",
    "PRJNA700657": "#9bd7d7"
}

unique_sites = network_df["site"].dropna().unique()
site_colors = px.colors.sample_colorscale(
    "turbo",
    [n/(len(unique_sites) -1) for n in range(len(unique_sites))]
)
site_mapping = dict(zip(unique_sites, site_colors))

### Electron flow

In [22]:
legend_dataset = [
    go.Scatter(
        mode="lines",
        x=[None],
        y=[None],
        marker=dict(size=10, color=color, symbol="square"),
        name=key,
        legendgroup="dataset",
        legendgrouptitle={
            "text": "Dataset (links)"
        }
    )
    for key, color in dataset_mapping.items()
]

legend_dataset.extend([
    go.Scatter(
        mode="markers",
        x=[None],
        y=[None],
        marker=dict(size=10, color=color, symbol="square"),
        name=key,
        legendgroup="cycle",
        legendgrouptitle={
            "text": "Cycle (nodes)"
        }
    )
    for key, color in cycle_mapping_legend.items()
])

In [23]:
node_df = pd.DataFrame({"label": categories.categories})

for redox_col in ("donor", "acceptor"):
    node_df = pd.merge(
        left=node_df,
        right=network_df[[redox_col, "cycle"]].drop_duplicates(),
        left_on="label",
        right_on=redox_col,
        how="left"
    )
    node_df = node_df.drop(redox_col, axis=1)

node_df["cycle"] = node_df["cycle_x"].fillna(node_df["cycle_y"])
node_df = node_df.drop(["cycle_x", "cycle_y"], axis=1)

node_df["color"] = node_df["cycle"].map(cycle_mapping)

# Manually change some node colors
node_df.loc[
    node_df["label"] == "ETC",
    "color"
] = "silver"

node_df.loc[
    node_df["label"] == "fumarate/succinate",
    "color"
] = "#545454"

node_df = node_df.drop_duplicates("label")
node_df.head()

,label,cycle,color
0,N2,nitrogen,#5271ff
1,SeO4,selenium,plum
2,H+,hydrogen,lightblue
3,F420,hydrogen,lightblue
4,trithionate,sulfur,#fcd400


In [24]:
group_col = "dataset"

network_df_grouped = network_df\
    .groupby([group_col, "donor", "acceptor"], as_index=False, observed=True)\
    .sum("size")

# TODO: color links according to cycle or sample type
network_df_grouped["color"] = network_df_grouped[group_col]\
    .map(dataset_mapping)
network_df_grouped["color"] = network_df_grouped["color"].fillna("darkgrey")

fig = go.Figure(
    go.Sankey(
        valuesuffix=" hits",
        arrangement="snap",
        node={
            # WARNING!!!!! Be careful of sorting of labels
            "label": categories.categories,
            # "x": node_df.index.tolist(),
            # "y": unique_redox_df["redox_potential_rev"].tolist(),
            "align": "center",
            "color": node_df["color"].tolist(),
            "line": dict(color="black", width=0.5),
            "pad": 10
        },
        link={
            "source": network_df_grouped["donor"].cat.codes.tolist(),
            "target": network_df_grouped["acceptor"].cat.codes.tolist(),
            "value":  network_df_grouped["size"].tolist(),
            "color":  network_df_grouped["color"].tolist(),
            "label": network_df_grouped["dataset"].tolist(),
            # "customdata": network_df_grouped[group_col].tolist(),
            # "hovertemplate": "%{customdata}",
            "line": dict(width=0.001),
            "arrowlen": 15
        }
    )
)

for trace in legend_dataset:
    fig.add_trace(trace)

fig.update_layout(
    width=1000,
    height=850,
    font=dict(
        size=15,
        family="Arial"
    ),
    # legend=dict(
    #     yanchor="top",
    #     # y=0.99,
    #     xanchor="right",
    #     # x=0.01
    # ),
    # template="simple_white",
    # paper_bgcolor="rgba(0,0,0,0)",
    # plot_bgcolor="rgba(0,0,0,0)"
    paper_bgcolor="#f8f7f2",
    plot_bgcolor="#f8f7f2"
)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

fig.update_traces(domain_y=list([0,1.0]), selector=dict(type='sankey'))

fig.write_image(
    os.path.join(
        RUN_DIR,
        f"closing-circuit-sankey-by-{group_col}-simple.svg"
    ),
    scale=20
)
fig.show()